# Hate Crime Undercount

Package versions pinned in `requirements.txt`: pandas==2.2.3, matplotlib==3.9.2, requests==2.32.3, jupyter==1.1.1, nbconvert==7.16.4, nbclient==0.10.0, ipykernel==6.29.5, pytest==8.3.3.

This notebook compares two federal views of hate crime: FBI UCR statistics based on voluntary law-enforcement reporting and BJS NCVS estimates based on household victimization surveys. The goal is to build a short, source-traceable data story about how many estimated hate crime victimizations make it into FBI-published statistics.

## 1. The two sources

This section identifies the source material and the unit differences that govern the analysis. NCVS counts victimizations among people age 12 or older living in households, while FBI UCR counts incidents, offenses, victims, and offenders as reported and classified by law enforcement. FBI data can include crimes against businesses, organizations, institutions, and society; NCVS does not cover those populations.

In [ ]:
from pathlib import Path
from textwrap import fill

import matplotlib.pyplot as plt
import pandas as pd
import requests

PROJECT = Path.cwd()
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
FIGURES = PROJECT / "figures"
for directory in [RAW, PROCESSED, FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)

FBI_TABLE1_URL = "https://ucr.fbi.gov/hate-crime/2013/tables/1tabledatadecpdf"
FBI_TABLE4_URL = "https://ucr.fbi.gov/hate-crime/2013/tables/4tabledatadecpdf"
BJS_BRIEFING_URL = "https://bjs.ojp.gov/content/pub/pdf/hcs1317pp.pdf"
FBI_MIRROR_TABLE1_URL = "https://raw.githubusercontent.com/emorisse/FBI-Hate-Crime-Statistics/master/2013/table1.csv"
FBI_MIRROR_TABLE4_URL = "https://raw.githubusercontent.com/emorisse/FBI-Hate-Crime-Statistics/master/2013/table4.csv"

source_overview = pd.DataFrame([
    {"source": "FBI UCR Hate Crime Statistics 2013", "unit": "incidents, offenses, victims, offenders", "coverage": "Voluntary law-enforcement reports; includes persons, organizations, businesses, institutions, and society", "url": FBI_TABLE1_URL},
    {"source": "BJS NCVS Hate Crime Statistics briefing", "unit": "victimizations and percentages", "coverage": "Survey of people age 12 or older living in households; includes reported and unreported victimizations", "url": BJS_BRIEFING_URL},
])
source_overview

## 2. Loading and validation

This section loads committed raw source values, verifies the expected schemas and anchor values, and records the source URLs used later in the notebook. The FBI GitHub mirror URLs from the project brief are retained as attempted mirror references, but the needed Table 1 and Table 4 values are sourced from the official FBI UCR pages because the mirror exposes only tables 13 and 14 for 2013.

In [ ]:
def require_columns(df: pd.DataFrame, columns: list[str], name: str) -> None:
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")


def require_values(actual, expected, name: str) -> None:
    actual_list = list(actual)
    if actual_list != expected:
        raise ValueError(f"{name} expected {expected}, got {actual_list}")


def fetch_if_missing(url: str, path: Path) -> None:
    if path.exists():
        return
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    path.write_bytes(response.content)


bjs_raw = pd.read_csv(RAW / "bjs_published_values.csv")
fbi_table1 = pd.read_csv(RAW / "fbi_2013_table1.csv")
fbi_table4 = pd.read_csv(RAW / "fbi_2013_table4.csv")

require_columns(bjs_raw, ["series", "measure", "display_label", "value", "unit", "ci_low", "ci_high", "ci_note", "source_url", "source_note"], "BJS raw values")
require_columns(fbi_table1, ["bias_category", "incidents", "offenses", "victims", "source_table", "source_url"], "FBI Table 1")
require_columns(fbi_table4, ["bias_category", "total_offenses", "source_table", "source_url"], "FBI Table 4")

total = fbi_table1.set_index("bias_category").loc["Total"]
assert int(total["incidents"]) == 5928
assert int(total["offenses"]) == 6933
assert int(total["victims"]) == 7242
assert int(fbi_table4.set_index("bias_category").loc["Total", "total_offenses"]) == 6933

pd.DataFrame([
    {"check": "FBI Table 1 total incidents", "expected": 5928, "status": "pass"},
    {"check": "FBI Table 1 total offenses", "expected": 6933, "status": "pass"},
    {"check": "FBI Table 1 total victims", "expected": 7242, "status": "pass"},
    {"check": "FBI Table 4 total offenses", "expected": 6933, "status": "pass"},
])

## 3. The funnel

The headline comparison starts with the BJS annual average of estimated hate crime victimizations and follows the reporting/classification pipeline to the UCR average annual victim count. The loss labels show where estimated victimizations disappear before reaching the FBI-published series. Confidence intervals are shown for NCVS estimates where BJS Appendix Table 1 provides them; the UCR count has no applicable NCVS confidence interval.

In [ ]:
funnel = bjs_raw[bjs_raw["series"] == "funnel"].copy()
funnel["step_order"] = range(1, len(funnel) + 1)
funnel = funnel.rename(columns={"display_label": "step_label"})
funnel["value"] = funnel["value"].astype(int)
funnel["loss_from_previous"] = funnel["value"].shift(1) - funnel["value"]
funnel["loss_percent_from_previous"] = (funnel["loss_from_previous"] / funnel["value"].shift(1) * 100).round(1)
funnel.loc[funnel["step_order"] == 1, ["loss_from_previous", "loss_percent_from_previous"]] = pd.NA
funnel_out = funnel[["step_order", "step_label", "value", "unit", "ci_low", "ci_high", "loss_from_previous", "loss_percent_from_previous", "source_url", "source_note", "ci_note"]]

require_values(funnel_out["value"], [204600, 101900, 45600, 15200, 7500], "BJS funnel values")
if not funnel_out["value"].is_monotonic_decreasing:
    raise ValueError("Funnel values must be descending")
funnel_out.to_csv(PROCESSED / "bjs_funnel.csv", index=False)
funnel_out

In [ ]:
plt.style.use("seaborn-v0_8-white")
total = int(funnel_out.loc[funnel_out["step_order"] == 1, "value"].iloc[0])
police_confirmed = int(funnel_out.loc[funnel_out["step_order"] == 4, "value"].iloc[0])
reaches_ucr = int(funnel_out.loc[funnel_out["step_order"] == 5, "value"].iloc[0])
not_police_confirmed = total - police_confirmed
confirmed_not_ucr = police_confirmed - reaches_ucr
pie_values = [not_police_confirmed, confirmed_not_ucr, reaches_ucr]
pie_labels = [
    "Not reported, not identified,\nor not police-confirmed",
    "Police-confirmed,\nnot in UCR endpoint",
    "Reaches FBI/UCR\nendpoint",
]
pie_colors = ["#2f6f73", "#d9b44a", "#9b4f4f"]

fig, ax = plt.subplots(figsize=(9.5, 7.2))
wedges, _ = ax.pie(
    pie_values,
    startangle=105,
    counterclock=False,
    colors=pie_colors,
    wedgeprops={"edgecolor": "white", "linewidth": 2, "width": 0.52},
)

legend_labels = [
    f"{label.replace(chr(10), ' ')}: {value:,.0f} ({value / total:.1%})"
    for label, value in zip(pie_labels, pie_values)
]
ax.legend(
    wedges,
    legend_labels,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=9,
)
ax.text(0, 0.08, f"{reaches_ucr / total:.1%}", ha="center", va="center", fontsize=28, fontweight="bold", color="#9b4f4f")
ax.text(0, -0.12, "of estimated\nvictimizations\nreach UCR", ha="center", va="center", fontsize=10, color="#333333")
ax.set_title("Most estimated hate crime victimizations never reach FBI statistics", fontsize=15, fontweight="bold", loc="left")
ax.text(-1.2, -1.24, "Base: 204,600 average annual NCVS hate crime victimizations, 2013-2017.\nSources: BJS NCVS and FBI UCR. UCR endpoint is victims, not victimizations.", fontsize=9, color="#444444")
fig.tight_layout()
fig.savefig(FIGURES / "funnel.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. The bias breakdown

This comparison uses BJS Appendix Table 2 percentages for 2013-2017. Categories can overlap because a victimization or victim may involve more than one bias motivation, so the bars should be read as category shares rather than pieces of a single 100% total.

In [ ]:
ncvs = bjs_raw[bjs_raw["series"] == "bias_breakdown"].set_index("measure")
ucr = bjs_raw[bjs_raw["series"] == "bias_breakdown_ucr"].set_index("measure")
bias_order = ["race_or_ethnicity", "religion", "gender", "sexual_orientation", "disability"]
bias_rows = []
for bias_type in bias_order:
    bias_rows.append({
        "bias_type": bias_type,
        "display_label": ncvs.loc[bias_type, "display_label"],
        "ncvs_percent": float(ncvs.loc[bias_type, "value"]),
        "ucr_percent": float(ucr.loc[bias_type, "value"]),
        "gap_percentage_points": round(float(ncvs.loc[bias_type, "value"]) - float(ucr.loc[bias_type, "value"]), 1),
        "source_url": BJS_BRIEFING_URL,
        "source_note": "Appendix table 2 and Figure 2",
    })
bias = pd.DataFrame(bias_rows)

expected_pairs = {
    "race_or_ethnicity": (57.2, 59.8),
    "religion": (7.9, 19.2),
    "gender": (27.2, 1.8),
    "sexual_orientation": (25.7, 17.7),
    "disability": (16.0, 1.4),
}
for bias_type, (ncvs_value, ucr_value) in expected_pairs.items():
    row = bias.set_index("bias_type").loc[bias_type]
    assert row["ncvs_percent"] == ncvs_value
    assert row["ucr_percent"] == ucr_value
bias.to_csv(PROCESSED / "bias_breakdown.csv", index=False)
bias

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.8))
positions = list(range(len(bias)))
width = 0.36
ax.bar([p - width / 2 for p in positions], bias["ncvs_percent"], width=width, label="NCVS", color="#2f6f73")
ax.bar([p + width / 2 for p in positions], bias["ucr_percent"], width=width, label="UCR", color="#d9b44a")

for p, row in zip(positions, bias.itertuples()):
    ax.text(p - width / 2, row.ncvs_percent + 1.3, f"{row.ncvs_percent:.1f}%", ha="center", fontsize=9)
    ax.text(p + width / 2, row.ucr_percent + 1.3, f"{row.ucr_percent:.1f}%", ha="center", fontsize=9)

ax.set_title("Gender and disability shares are much lower in UCR than in NCVS", fontsize=14, fontweight="bold", loc="left")
ax.set_ylabel("Percent of hate crime victimizations/victims")
ax.set_xticks(positions)
ax.set_xticklabels([fill(label, 14) for label in bias["display_label"]])
ax.set_ylim(0, 68)
ax.legend(frameon=False, ncols=2, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
ax.text(0, -0.2, "Categories can overlap; values do not sum to 100%. Source: BJS Appendix Table 2, NCVS and UCR, 2013-2017.", transform=ax.transAxes, fontsize=9, color="#444444")
fig.tight_layout()
fig.savefig(FIGURES / "bias_breakdown.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Findings and caveats

This final section creates the FBI 2013 summary file, renders a compact table of key numbers, and states the methodological caveats needed for the video story. The comparison is intentionally framed as a gap between federal data systems, not as a perfect unit-for-unit equivalence.

In [ ]:
fbi_summary = fbi_table1[["bias_category", "incidents", "offenses", "victims", "source_table", "source_url"]].copy()
fbi_summary.insert(0, "year", 2013)
if (fbi_summary[["incidents", "offenses", "victims"]] < 0).any().any():
    raise ValueError("FBI counts must be non-negative")
fbi_summary.to_csv(PROCESSED / "fbi_2013_summary.csv", index=False)

summary_table = pd.DataFrame([
    {"measure": "NCVS 2013 total hate crime victimizations", "value": 254900, "unit": "victimizations", "source": "BJS Appendix table 3"},
    {"measure": "NCVS 2013 reported to police", "value": 88400, "unit": "victimizations", "source": "BJS Appendix table 3"},
    {"measure": "NCVS 2013 not reported to police", "value": 154300, "unit": "victimizations", "source": "BJS Appendix table 3"},
    {"measure": "BJS 2013-2017 annual average victimizations", "value": 204600, "unit": "victimizations", "source": "BJS Appendix table 1"},
    {"measure": "BJS 2013-2017 police-confirmed by victim report", "value": 15200, "unit": "victimizations", "source": "BJS Appendix table 1"},
    {"measure": "UCR 2013-2017 annual average hate crime victims", "value": 7500, "unit": "victims", "source": "BJS Appendix table 1"},
    {"measure": "FBI UCR 2013 total incidents", "value": 5928, "unit": "incidents", "source": "FBI Table 1"},
    {"measure": "FBI UCR 2013 total victims", "value": 7242, "unit": "victims", "source": "FBI Table 1"},
])
summary_table["value"] = summary_table["value"].map(lambda value: f"{value:,.0f}")
summary_table

### Short writeup

BJS estimates an average of 204,600 hate crime victimizations per year in 2013-2017, but the comparable UCR endpoint is about 7,500 hate crime victims per year. In the funnel, about half of estimated victimizations are reported to police (101,900), fewer than a quarter involve victims telling police they believed it was a hate crime (45,600), and only 15,200 involve victims saying police confirmed the hate motivation. That means the FBI-published endpoint is roughly 3.7% of the starting NCVS estimate.

The bias breakdown shows broad agreement on race or ethnicity, but large gaps for gender and disability. In BJS Appendix Table 2, NCVS estimates 27.2% of hate crime victimizations involved gender bias versus 1.8% in UCR, and 16.0% involved disability bias versus 1.4% in UCR. Religion moves the other direction: 7.9% in NCVS versus 19.2% in UCR.

These numbers are not strict apples-to-apples counts. NCVS counts victimizations, so one person can be victimized more than once; FBI UCR counts incidents, offenses, and victims as reported and classified by police. NCVS covers people age 12 or older in households and excludes businesses, organizations, institutions, and military barracks. FBI hate crime reporting is voluntary, and many jurisdictions report no hate crimes or do not participate fully. The funnel should therefore be read as evidence of a reporting and classification gap between systems, not as a precise conversion rate from victimizations to incidents.